<h1 align="center">Laboratorio 4</h1>

## Información

**Integrantes:**

| Name              | Institution ID | GitHub User |
| ----------------- | -------------- | ----------- |
| Josué Say         | 22801          | JosueSay    |
| Carlos Valladares | 221164         | vgcarlol    |

- [Repositorio](https://github.com/JosueSay/intro-to-computer-vision/tree/main/labs/lab4)

## Preparación de entorno

In [ ]:
# %pip install -r requirements.txt
# jupyter nbconvert lab4.ipynb --to html

## Librerías y Configuración

In [ ]:
import cv2, math, os, time
import numpy as np
import matplotlib.pyplot as plt
from tabulate import tabulate

## Task 1

El objetivo es evaluar la comprensión de la geometría proyectiva y la manipulación algebraica de coordenadas homogéneas. Entonces, responda las siguientes preguntas demostrando el desarrollo matemático. No se aceptan respuestas puramente textuales sin respaldo algebraico.

### Inciso 1

Una homografía $H$ es una matriz de $3 \times 3$. Explique matemáticamente por qué, aunque tiene 9 elementos, solo posee 8 grados de libertad (GDL)

#### Inciso a

Adicionalmente, responda. Si tuviéramos una cámara que solo rota sobre su eje óptico (sin traslación ni cambio de perspectiva), ¿la matriz de transformación sigue teniendo 8 GDL o se reduce? Demuestre la estructura de dicha matriz simplificada.

### Inciso 2

En el algoritmo DLT (Direct Linear Transform), convertimos el problema $x' = Hx$ en un sistema de la forma $Ah = 0.$

Explique por qué buscamos el vector singular asociado al menor valor singular de $A$ en lugar de simplemente invertir la matriz. ¿Qué representa geométricamente ese "menor valor singular" cuando los datos tienen ruido?

### Inciso 3

Si usted selecciona 4 puntos para calcular $H$, pero 3 de ellos son colineales (están en la misma línea recta), el algoritmo fallará.

Explique algebraicamente qué le sucede a la matriz $A$ del sistema DLT en este caso y por qué no tiene solución única.

## Task 2

El objetivo de esta parte es implementar el pipeline de alineación sin depender de cajas negras. Por ello considere que no deben de usar cv2.findHomography o cv2.RANSAC. Además para este laboratorio necesitará crear su propio dataset, por ello tome 3 fotografías propias de una escena planar (i.e. una pancarta en una pared, un cuadro, o una fachada de edificio lejana) con ángulos y perspectivas drásticamente diferentes. Con esto realice:

### Inciso 1 - Detección y Macheo

#### Inciso a

Utilice SIFT u ORB (permitido usar OpenCV aquí) para detectar puntos de interés y descriptores.

#### Inciso b

Realice un emparejamiento de fuerza bruta (Brute-Force Matcher).

#### Inciso c

Requisito: Visualice los matches antes de filtrar. Debe verse una cantidad considerable de ruido/errores.

### Inciso 2 - Algoritmo DLT

#### Inciso a

Escriba una función calcular_homografia_dlt(puntos_src, puntos_dst) que reciba exactamente 4 pares de puntos.

#### Inciso b

Debe construir la matriz $A$ de tamaño $8 \times 9$.

#### Inciso c

Debe resolver el sistema usando SVD (numpy.linalg.svd).

#### Inciso d

Nota: Debe normalizar los puntos antes de entrar al DLT (restar la media y dividir por la desviación estándar) para estabilidad numérica, y des-normalizar la matriz $H$ resultante.

### Inciso 3 - RANSAC Manual

#### Inciso a

Implemente la función ransac_homografia(matches, umbral, prob_exito).

#### Inciso b

Cálculo de N: Su código debe calcular dinámicamente el número de iteraciones $N$ basado en la fórmula de probabilidad vista en clase. No "hardcodee" el número 1000.

#### Inciso c - Bucle

##### Inciso i

Seleccione 4 matches aleatorios.

##### Inciso ii

Llame a su función DLT.

##### Inciso iii

Proyecte todos los puntos fuente usando H_test.

##### Inciso iv

Calcule el error de reproyección (distancia Euclidiana) y cuente los inliers.

#### Inciso d

Refinamiento: Una vez encontrado el mejor conjunto de inliers, recalcule $H$ final usando todos los inliers (no solo los 4 iniciales) mediante SVD.

> La salida esperada es una imagen “stitched” (panorama) mostrando la alineación correcta.

## Task 3

En esta parte lo que se busca es evaluar el criterio profesional ante situaciones adversas y trade-offs de diseño. Para ello realice y responda lo siguiente

### Inciso 1

Ejecute su algoritmo RANSAC variando el parámetro de umbral de error (threshold) en pixeles (ej. 1px, 5px, 20px)

#### Inciso a

Genere una gráfica: Eje X = Umbral, Eje Y = Número de Inliers encontrados.

#### Inciso b

Discusión: Como ingeniero, ¿qué riesgo corre si establece un umbral demasiado estricto (ej. 0.5px)? ¿Qué pasa con la matriz final si el umbral es muy laxo (ej. 50px)?

### Inciso 2

Imagine que usted es el Lead Computer Vision Engineer de una empresa de drones. Deben alinear imágenes térmicas de paneles solares tomadas desde el aire para detectar fallos.

#### Inciso a

Problema: El drone vuela a 50 metros de altura. El terreno no es perfectamente plano (hay colinas suaves), pero los paneles sí son planos.

#### Inciso b

Pregunta A: ¿Es válido usar una Homografía global para unir todo el mapa del terreno? ¿Por qué sí o por qué no?

#### Inciso c

Pregunta B: Su algoritmo RANSAC está tardando demasiado (3 segundos por frame) en la computadora a bordo del drone (Raspberry Pi). La telemetría indica que el 90% de los matches iniciales son outliers debido al pasto y árboles repetitivos.

##### Inciso i

Proponga una estrategia concreta para reducir el tiempo de ejecución sin cambiar el hardware. (Pista: Piense en la fórmula de N o en pre-filtrado geométrico)